# Agent benchmark API

- This notebook explores `agent_benchmark_utils`, a simulation of picking the
  best of K agent configs on a benchmark of N tasks run with S seeds each
- The running example is a small benchmark with 4 configs, 30 tasks, and 3
  seeds, small enough to look at every array
- The pedagogical arc:
  - Primitives: the model parameters, the draws, and the true resolve rate
    $\bar{Y}_A$
  - Composition: one benchmark run, the winner, and its winner's curse
  - Patterns: re-evaluating the winner, replications, and bootstrap intervals

In [1]:
%load_ext autoreload
%autoreload 2

import logging

import numpy as np

In [2]:
import helpers.hintrospection as hintros
import helpers.hnotebook as hnotebook

import agent_benchmark_utils as utils

# Initialize notebook configuration and logging.
hnotebook.config_notebook()
_LOG = logging.getLogger(__name__)
utils.init_loggers(_LOG)

# Convert `display` into `print()` when running outside IPython.
try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore

## Library Overview

- **What problem it solves**: a benchmark score mixes skill and luck, so the
  config with the best score is not always the best config, and its score is
  too high on average (the winner's curse)
- **Key abstraction**: a `World`, i.e., one draw of config skills, tasks, and
  seed outcomes, from which scores, winners, and errors are computed
- **Mental model**:

| Object | Description | Comments |
|--------|-------------|----------|
| `SimParams(...)` | Sizes K, N, S and the spreads of the model | Frozen dataclass |
| `draw_world(params, rng)` | One benchmark run | Returns a `World` |
| `World.ybar` | True resolve rate $\bar{Y}_A$ of each config | Shape `(K,)` |
| `World.y` | 0/1 outcomes | Shape `(K, N, S)` |
| `compute_scores(y)` | Observed score of each config | Mean over tasks and seeds |
| `run_replications(params, n_reps, seed)` | Many independent runs | One row of metrics per run |
| `compute_cis(y, n_boot, method, rng)` | Bootstrap interval per config | Methods `task`, `task_seed` |

- **Key classes**:
  - `SimParams`: the parameters of the simulated benchmark
  - `World`: the arrays of one benchmark run
- **Notation** of the white paper (section "Skill-Luck Decomposition"):
  - $S_A$: the strategy of agent $A$, here one config $k$
  - $Y$: the `resolved` outcome of a config on one task and one seed
  - $\bar{Y}_A = E[Y \mid S_A, I_A]$: the true resolve rate `ybar` of a config
  - $L_A(Y) = Y - \bar{Y}_A$: the luck component of the outcome
  - $\theta_k$, $b_i$, $u_{ki}$ are parameters of this simulation, not
    symbols of the paper

# Part 1: The model

## Cell 1.1: Model parameters

- Config $k$ has skill $\theta_k \sim N(\mu_\theta, \sigma_\theta^2)$
- Task $i$ has difficulty $b_i \sim N(\mu_b, \sigma_b^2)$
- Config $k$ and task $i$ have an interaction $u_{ki} \sim N(0, \sigma_u^2)$
- Each seed is a Bernoulli draw with
  $P(\text{resolve}) = \text{sigmoid}(\theta_k - b_i + u_{ki})$

In [3]:
# Build the parameters of the running example.
params = utils.SimParams(n_configs=4, n_tasks=30, n_seeds=3)
print("params=", params)
# Outcome: the sizes are set, and the spreads keep their defaults.

params= SimParams(n_configs=4, n_tasks=30, n_seeds=3, mu_theta=1.0, sigma_theta=0.2, mu_b=0.0, sigma_b=2.0, sigma_u=1.0)


In [4]:
# Show the default parameters, which are assumptions and not estimates.
print("utils.SimParams()=", utils.SimParams())
# Outcome: 10 configs, 100 tasks, 3 seeds, with configs close together
# (`sigma_theta=0.2`) and tasks far apart (`sigma_b=2.0`).

utils.SimParams()= SimParams(n_configs=10, n_tasks=100, n_seeds=3, mu_theta=1.0, sigma_theta=0.2, mu_b=0.0, sigma_b=2.0, sigma_u=1.0)


## Cell 1.2: Drawing skills, difficulties, and interactions

In [5]:
# Draw the skill of each config.
seed = 0
rng = np.random.default_rng(seed)
theta = utils.draw_skills(params, rng)
print("theta=", theta)
# Outcome: 4 skills close to `mu_theta=1.0`.

theta= [1.02514604 0.97357903 1.12808453 1.02098002]


In [6]:
# Draw the difficulty of each task.
b = utils.draw_difficulties(params.n_tasks, params, rng)
print("b.shape=", b.shape)
print("b[:5]=", b[:5])
# Outcome: 30 difficulties spread much more widely than the skills.

b.shape= (30,)
b[:5]= [-1.07133875  0.72319011  2.60800009  1.89416193 -1.40747047]


In [7]:
# Draw the config-by-task interactions.
u = utils.draw_interactions(params.n_configs, params.n_tasks, params, rng)
print("u.shape=", u.shape)
# Outcome: one interaction per (config, task), so a config can be lucky on
# some tasks and not on others.

u.shape= (4, 30)


In [8]:
# Draw the seed outcomes from the solve probabilities.
p = utils.sigmoid(theta[:, None] - b[None, :] + u)
y = utils.draw_outcomes(p, params.n_seeds, rng)
print("y.shape=", y.shape)
# Outcome: a 0/1 array of shape (K, N, S).

y.shape= (4, 30, 3)


## Cell 1.3: The true resolve rate

- $\bar{Y}_A$ is the expected outcome of a config over the whole task
  population and the seeds, not over the 30 tasks drawn
- It is computed by numerical integration, so it has no sampling noise

In [9]:
# Compute the true resolve rate of each config.
ybar = utils.compute_true_resolve_rate(theta, params)
print("ybar=", ybar)
# Outcome: rates around 0.64, in the same order as `theta`.

ybar= [0.64158082 0.63475277 0.65507017 0.64103089]


In [10]:
# Check the symmetry case: `theta = mu_b` gives exactly 0.5.
ybar_sym = utils.compute_true_resolve_rate(np.array([params.mu_b]), params)
print("ybar_sym=", ybar_sym)
# Outcome: 0.5.

ybar_sym= [0.5]


## Cell 1.4: One benchmark run

In [11]:
# Draw a whole benchmark run in one call.
world = utils.draw_world(params, np.random.default_rng(seed))
print("world.ybar=", world.ybar)
print("world.y.shape=", world.y.shape)
# Outcome: the same arrays as in the cells above, bundled in a `World`.

world.ybar= [0.64158082 0.63475277 0.65507017 0.64103089]
world.y.shape= (4, 30, 3)


In [12]:
# Convert the outcomes into the generic long table.
df_outcomes = utils.outcomes_to_frame(world.y)
display(df_outcomes.head())
# Outcome: one row per (config, task, seed) with columns `config`, `task`,
# `seed`, `resolved`.

,config,task,seed,resolved
0,0,0,0,1
1,0,0,1,1
2,0,0,2,1
3,0,1,0,1
4,0,1,1,1


In [13]:
hintros.print_obj_info(utils.draw_world)

- `draw_world(params: agent_benchmark_utils.SimParams, rng: numpy.random._generator.Generator) -> agent_benchmark_utils.World`  
  Draw configs, tasks, interactions, and outcomes for one benchmark run...

https://github.com/gpsaggese/gpsaggese.github.io/blob/UmdTask537_Review_students_research_projects/research/Causal_Analysis_of_Agent_Skill_And_Luck/agent_benchmark_utils.py#L253


# Part 2: Scores and the winner's curse

## Cell 2.1: Observed scores and the winner

In [14]:
# Compute the observed score of each config.
scores = utils.compute_scores(world.y)
print("scores=", scores)
# Outcome: the mean over tasks and seeds, which mixes skill $\bar{Y}_A$ and
# luck $L_A(Y)$.

scores= [0.65555556 0.74444444 0.72222222 0.66666667]


In [15]:
# Pick the config with the best observed score.
winner = utils.select_winner(scores)
print("winner=", winner)
# Outcome: the index of the highest score.

winner= 1


In [16]:
# Check whether the winner is also the truly best config.
truly_best = int(np.argmax(world.ybar))
print("truly_best=", truly_best)
# Outcome: with configs this close, the two can differ. Here the winner is
# config 1 and the truly best is config 2.

truly_best= 2


## Cell 2.2: Ranks and the curse

In [17]:
# Compare the observed ranks with the true ranks.
print("observed ranks=", utils.compute_ranks(scores))
print("true ranks=", utils.compute_ranks(world.ybar))
# Outcome: rank 1 is the best. Here the observed winner, config 1, has the
# worst true rate of the 4.

observed ranks= [4. 1. 2. 3.]
true ranks= [2. 4. 1. 3.]


In [18]:
# Measure the winner's curse of this run.
curse = scores[winner] - world.ybar[winner]
print("curse=", curse)
# Outcome: the winner's observed score minus its true rate. One run is noisy;
# Part 4 averages many runs.

curse= 0.10969167236336053


# Part 3: Re-evaluating the winner

## Cell 3.1: Fresh seeds on the same tasks

- Rerunning the winner with new seeds removes the seed luck
- It keeps the luck of which tasks were in the benchmark, including the
  interactions $u_{ki}$ of the winner with those tasks

In [19]:
# Rerun the winner on the same tasks with new seeds.
fresh_seed = 1
fresh_seed_score = utils.evaluate_on_fresh_seeds(
    world, winner, params.n_seeds, np.random.default_rng(fresh_seed)
)
print("fresh_seed_score=", fresh_seed_score)
# Outcome: a new score for the winner on the same 30 tasks, lower than its
# observed score but still above its true rate.

fresh_seed_score= 0.6888888888888889


## Cell 3.2: Fresh tasks

- Running the winner on new tasks does not reuse anything that selected it

In [20]:
# Run the winner on new tasks with the same budget.
fresh_task_seed = 2
fresh_task_score = utils.evaluate_on_fresh_tasks(
    world.theta[winner], params, np.random.default_rng(fresh_task_seed)
)
print("fresh_task_score=", fresh_task_score)
# Outcome: a score that is unbiased for the winner's true rate. In this one
# run it happens to equal the fresh-seed score; Part 4 averages many runs.

fresh_task_score= 0.6888888888888889


# Part 4: Replications

## Cell 4.1: One replication

In [21]:
# Run one replication and collect its metrics.
metrics = utils.run_replication(params, np.random.default_rng(seed))
print("metrics=", metrics)
# Outcome: the curse, whether the winner is truly best, the regret, the rank
# error, and the two re-evaluation errors.

metrics= {'curse': np.float64(0.10969167236336053), 'is_truly_best': 0.0, 'regret': np.float64(0.020317394638943775), 'rank_error': np.float64(1.5), 'fresh_seed_error': np.float64(0.13191389458558278), 'fresh_task_error': np.float64(-0.012530549858861706)}


## Cell 4.2: Many replications

In [22]:
# Run 500 independent replications.
n_reps = 500
df_reps = utils.run_replications(params, n_reps, seed)
display(df_reps.head())
# Outcome: one row of metrics per replication.

,curse,is_truly_best,regret,rank_error,fresh_seed_error,fresh_task_error
0,-0.020161,1.0,0.000000,1.00,0.002062,0.002062
1,0.007322,0.0,0.003326,1.00,-0.014900,0.029545
2,0.053951,1.0,0.000000,0.75,0.053951,-0.001604
3,-0.007107,0.0,0.027206,1.00,-0.051551,-0.073773
4,0.023536,0.0,0.012738,0.75,-0.065353,-0.076464


In [23]:
# Average the metrics and add Monte Carlo standard errors.
summary = utils.summarize_replications(df_reps, params)
print("summary=", summary)
# Outcome:
# - `curse` is positive, about 3.9 points
# - `fresh_seed_error` is between 0 and the curse
# - `fresh_task_error` is within 2 standard errors of 0

summary= {'curse': np.float64(0.038664871406624536), 'curse_se': np.float64(0.0026902178320578575), 'is_truly_best': np.float64(0.446), 'is_truly_best_se': np.float64(0.02225215307859594), 'regret': np.float64(0.014173218587634042), 'regret_se': np.float64(0.0008701372733304572), 'rank_error': np.float64(0.892), 'rank_error_se': np.float64(0.022037315775138), 'fresh_seed_error': np.float64(0.008375982517735647), 'fresh_seed_error_se': np.float64(0.0031648232574328986), 'fresh_task_error': np.float64(-0.0041351285933754625), 'fresh_task_error_se': np.float64(0.002817685622881165), 'extra_runs': 90.0, 'extra_cost_frac': 0.25}


In [24]:
hintros.print_obj_info(utils.run_replication)

- `run_replication(params: agent_benchmark_utils.SimParams, rng: numpy.random._generator.Generator) -> Dict[str, float]`  
  Draw one benchmark, select the winner, and measure what went wrong...

https://github.com/gpsaggese/gpsaggese.github.io/blob/UmdTask537_Review_students_research_projects/research/Causal_Analysis_of_Agent_Skill_And_Luck/agent_benchmark_utils.py#L414


# Part 5: Bootstrap intervals

## Cell 5.1: Bootstrap replicates of one score

- `task`: resample tasks, keeping all the seeds of each task together
- `task_seed`: resample tasks, then resample seeds inside each drawn task

In [25]:
# Print the available resampling schemes.
print("utils.BOOTSTRAP_METHODS=", utils.BOOTSTRAP_METHODS)
# Outcome: `task` and `task_seed`.

utils.BOOTSTRAP_METHODS= ['task', 'task_seed']


In [26]:
# Compute bootstrap replicates of the score of config 0.
n_boot = 1000
method = "task"
replicates = utils.bootstrap_scores(
    world.y[0], n_boot, method, np.random.default_rng(seed)
)
print("replicates.std()=", replicates.std())
# Outcome: the bootstrap standard error of one config's score.

replicates.std()= 0.0650818810005358


## Cell 5.2: One interval per config

In [27]:
# Compute a 95% interval for every config.
df_cis = utils.compute_cis(world.y, n_boot, method, np.random.default_rng(seed))
df_cis["ybar"] = world.ybar
display(df_cis)
# Outcome: one interval per config, next to the true rate it should contain.

,config,score,lo,hi,ybar
0,0,0.655556,0.522222,0.766667,0.641581
1,1,0.744444,0.622222,0.855556,0.634753
2,2,0.722222,0.588889,0.833333,0.655070
3,3,0.666667,0.544444,0.800000,0.641031


## Cell 5.3: Coverage of the intervals

In [28]:
# Measure how often each scheme covers the true rate.
n_coverage_reps = 50
n_coverage_boot = 200
df_cov = utils.run_coverage(params, n_coverage_reps, n_coverage_boot, seed)
display(df_cov)
# Outcome: coverage of both schemes over 200 intervals, close to 95%. The
# example notebook shows the full grid.

,method,coverage,coverage_se,mean_width,n_intervals
0,task,0.950,0.015411,0.258236,200
1,task_seed,0.965,0.012995,0.284900,200


## Summary: The Mental Model

- A benchmark score is the true resolve rate $\bar{Y}_A$ plus luck
  $L_A(Y)$, and the luck comes from the tasks drawn, the config-by-task
  interactions, and the seeds
- Picking the best observed score selects for luck, so the winner's score is
  too high on average: the winner's curse
- Rerunning the winner on the same tasks with new seeds removes only part of
  the curse; running it on fresh tasks removes all of it
- Bootstrap intervals that resample tasks cover the true rate close to 95% of
  the time; also resampling seeds inside each task counts seed noise twice
  when seeds are independent, so its intervals are wider than needed